In [ ]:
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
import os
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
base_dir   = '/kaggle/input/datasets/raphalperon/data-scap/patients4/'
output_dir = '/kaggle/working/'


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
import os
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
base_dir   = '/kaggle/input/datasets/raphalperon/data-scap/patient/patient/'
output_dir = '/kaggle/working/'

In [ ]:

# Étape 1 : Pré-traitement et extraction de patches
# Cette étape convertit les volumes 3D en petits patches pour l'entraînement du modèle.
patches_ct = []
patches_masque = []
patients_valides  = 0

for dossier in sorted(os.listdir(base_dir)):
    dossier_path = os.path.join(base_dir, dossier)
    if not os.path.isdir(dossier_path):
        continue
        
    ct_path = os.path.join(dossier_path, 'ct.nii')
    if not os.path.exists(ct_path):
        continue
        
    # Chargement du scanner et conversion en float32 pour le calcul
    ct = nib.load(ct_path).get_fdata().astype(np.float32)

    # Normalisation : on se focalise sur les tissus mous et l'os (-200 à 1800 HU)
    ct = np.clip(ct, -200, 1800)
    ct = (ct - (-200)) / (1800 - (-200))

    # Fusion des masques gauche et droit pour obtenir une segmentation globale
    masque_global = np.zeros(ct.shape, dtype=np.uint8)
    for nom in ['scapula_left', 'scapula_right']:
        masque_path = os.path.join(dossier_path, 'segmentations', f'{nom}.nii')
        if os.path.exists(masque_path):
            masque_local = (nib.load(masque_path).get_fdata() > 0).astype(np.uint8)

            # Correction de l'orientation si nécessaire (différence entre CT (z,y,x) et NIfTI (x,y,z))
            if masque_local.shape != ct.shape:
                masque_local = np.transpose(masque_local, (2, 1, 0))
            if masque_local.shape == ct.shape:
                masque_global = np.clip(masque_global + masque_local, 0, 1)

    # On ignore les volumes dont la segmentation est trop faible (< 20k voxels)
    if masque_global.sum() < 20000:
        del ct, masque_global
        continue

    taille, pas = 64, 32
    z_max, y_max, x_max = ct.shape
    n = 0
    # Découpage en patches de 64x64x64 avec un chevauchement de 50% (stride 32)
    for z in range(0, z_max - taille + 1, pas):
        for y in range(0, y_max - taille + 1, pas):
            for x in range(0, x_max - taille + 1, pas):
                patch_masque = masque_global[z:z+taille, y:y+taille, x:x+taille]
                patch_ct= ct[z:z+taille,     y:y+taille, x:x+taille]
                # On garde le patch s'il contient de la scapula (masque) 
                # OU s'il contient de l'os dense (HU > 800) pour apprendre au modèle à ne pas confondre les structures.
                if patch_masque.sum() > 50 or patch_ct.max() > 0.5:
                    # Stockage en float16 pour optimiser l'occupation de la RAM
                    patches_ct.append(patch_ct.astype(np.float16)) 
                    patches_masque.append(patch_masque.astype(np.uint8))
                    n += 1

    patients_valides += 1
    print(f"{dossier} → {n} patches | total : {len(patches_ct)}")
    del ct, masque_global # Libération de la mémoire

print(f"\n {patients_valides} patients | {len(patches_ct)} patches")

# Sauvegarde du dataset compressé pour l'entraînement ultérieur
np.savez_compressed(
    output_dir + 'patches.npz',
    patches_ct_array = np.array(patches_ct, dtype=np.float16),
    patches_masque_array = np.array(patches_masque, dtype=np.uint8)
)
print(f"Sauvegardé | {os.path.getsize(output_dir+'patches.npz')/1e9:.1f} GB")

In [ ]:
#Étape 2 : Chargement du Dataset
# On recharge les patches précalculés pour l'entraînement du modèle.
# Note : On conserve le typage float16/uint8 pour limiter l'empreinte mémoire.
data    = np.load(output_dir + 'patches.npz')

# Chargement des images (CT) et des masques de segmentation
patches = data['patches_ct_array'].astype(np.float16)
masques = data['patches_masque_array'].astype(np.uint8)
print(f"Patches chargés : {patches.shape}")
print(f"Taille RAM : {patches.nbytes/1e9:.1f} GB")

In [ ]:
#Étape 3 : Entraînement du modèle UNet 3D

import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
import os
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import gc

base_dir   = '/kaggle/input/datasets/raphalperon/data-scap/patients4/'
output_dir = '/kaggle/working/'

#bloc double convolution classique pour l'extraction de caractéristiques spatiales.
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.encodeur = nn.ModuleList()
        self.pool     = nn.MaxPool3d(kernel_size=2, stride=2)
        # Partie descendante (Encoder)
        for feature in features:
            self.encodeur.append(DoubleConv(in_channels, feature))
            in_channels = feature
        self.fond = DoubleConv(features[-1], features[-1] * 2)
        self.decodeur_up   = nn.ModuleList()
        self.decodeur_conv = nn.ModuleList()

        # Partie ascendante (Decoder)
        for feature in reversed(features):
            self.decodeur_up.append(
                nn.ConvTranspose3d(feature * 2, feature, kernel_size=2, stride=2)
            )
            self.decodeur_conv.append(DoubleConv(feature * 2, feature))
        self.sortie  = nn.Conv3d(features[0], out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        skip_connections = []
        for couche in self.encodeur:
            x = couche(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.fond(x)
        skip_connections = skip_connections[::-1]
        for i in range(len(self.decodeur_up)):
            x    = self.decodeur_up[i](x)
            skip = skip_connections[i]
            # Ajustement de la taille si nécessaire pour la concaténation (skip connections)
            if x.shape != skip.shape:
                x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.decodeur_conv[i](x)
        return self.sigmoid(self.sortie(x))

class ScapulaDataset(Dataset):
    def __init__(self, patches, masques):
        self.patches = patches
        self.masques = masques

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        # Conversion en float32 pour PyTorch
        patch  = self.patches[idx].astype(np.float32).copy()
        masque = self.masques[idx].astype(np.float32).copy()

        for axe in [0, 1, 2]:
            if np.random.rand() > 0.5:
                # Data Augmentation : Flips aléatoires sur les 3 axes
                patch  = np.flip(patch,  axe).copy()
                masque = np.flip(masque, axe).copy()

        #Ajout d'un bruit gaussien léger pour la robustesse
        patch += np.random.normal(0, 0.02, patch.shape)
        patch  = np.clip(patch, 0, 1)

        return (torch.tensor(patch,  dtype=torch.float32).unsqueeze(0),
                torch.tensor(masque, dtype=torch.float32).unsqueeze(0))

# Loss fonction qui combine Dice Loss (pour la forme) et BCE pondéré (pour gérer le déséquilibre fond/scapula).
def bce_dice_loss(prediction, cible, epsilon=1e-6):
    p = prediction.view(-1) on applati le tenseur
    c = cible.view(-1)
    # Dice Loss
    intersection = (p * c).sum()
    dice = 1 - (2 * intersection + epsilon) / (p.sum() + c.sum() + epsilon)
    # Weighted BCE : Un voxel scapula est 20x plus important qu'un voxel de fond
    bce  = torch.nn.functional.binary_cross_entropy(
        prediction, cible, weight=(cible * 19 + 1) 
    )
    return dice + bce

#Configuration de l'entraînement

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

dataset    = ScapulaDataset(patches, masques)
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=4)

model      = UNet3D().to(device)
if torch.cuda.device_count() > 1:
    print(f"Utilisation de {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
optimiseur = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiseur, mode='min', patience=4, factor=0.5)

#Boucle d'entraînement
nb_epochs                = 25
patience                 = 5
train_losses             = []
val_losses               = []
meilleur_val             = float('inf')
epochs_sans_amelioration = 0

for epoch in range(nb_epochs):
    model.train()
    train_loss = 0
    for patch, masque in train_loader:
        patch, masque = patch.to(device), masque.to(device)
        prediction    = model(patch)
        loss          = bce_dice_loss(prediction, masque)
        optimiseur.zero_grad()
        loss.backward()
        optimiseur.step()
        train_loss += loss.item()
        # Nettoyage mémoire explicite pour le GPU
        del patch, masque, prediction, loss
        torch.cuda.empty_cache()
        
    train_loss /= len(train_loader)
    gc.collect()
    torch.cuda.empty_cache()
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for patch, masque in val_loader:
            patch, masque = patch.to(device), masque.to(device)
            val_loss     += bce_dice_loss(model(patch), masque).item()
    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}/{nb_epochs} | Train : {train_loss:.4f} | Val : {val_loss:.4f}")

    if val_loss < meilleur_val:
        meilleur_val             = val_loss
        epochs_sans_amelioration = 0
        torch.save(model.state_dict(), '/kaggle/working/meilleur_modele_scapula.pth')
        print(f"  Meilleur modèle sauvegardé ! (val={meilleur_val:.4f})")
    else:
        epochs_sans_amelioration += 1
        print(f"  Pas d'amélioration ({epochs_sans_amelioration}/{patience})")
        if epochs_sans_amelioration >= patience:
            print(f"  Early stopping à l'epoch {epoch+1}")
            break

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch')
plt.ylabel('BCE + Dice Loss')
plt.title("Courbe d'apprentissage")
plt.legend()
plt.grid(True)
plt.savefig('/kaggle/working/courbe_apprentissage.png')
plt.show()
print(f"\n  Meilleur val loss : {meilleur_val:.4f} — Dice : {1 - meilleur_val/2:.3f}")

In [2]:
#Étape 4 : Inférence complète et Post-traitement de la scapula
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
import os
import struct
from skimage.measure import marching_cubes
from scipy.ndimage import gaussian_filter, label, binary_fill_holes
import scipy.ndimage as ndimage

#Chemins
dossier_patients = '/kaggle/input/datasets/raphalperon/data-scap/patients4/patients3/patients3'
chemin_modele    = '/kaggle/input/datasets/raphalperon/data-scap/meilleur_modele_scapula2.pth'
patient          = 's0224'
dossier_sortie   = '/kaggle/working/'

#Modèle
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.encodeur = nn.ModuleList()
        self.pool     = nn.MaxPool3d(kernel_size=2, stride=2)
        for feature in features:
            self.encodeur.append(DoubleConv(in_channels, feature))
            in_channels = feature
        self.fond = DoubleConv(features[-1], features[-1] * 2)
        self.decodeur_up   = nn.ModuleList()
        self.decodeur_conv = nn.ModuleList()
        for feature in reversed(features):
            self.decodeur_up.append(
                nn.ConvTranspose3d(feature * 2, feature, kernel_size=2, stride=2)
            )
            self.decodeur_conv.append(DoubleConv(feature * 2, feature))
        self.sortie  = nn.Conv3d(features[0], out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        skip_connections = []
        for couche in self.encodeur:
            x = couche(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.fond(x)
        skip_connections = skip_connections[::-1]
        for i in range(len(self.decodeur_up)):
            x    = self.decodeur_up[i](x)
            skip = skip_connections[i]
            if x.shape != skip.shape:
                x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.decodeur_conv[i](x)
        return self.sigmoid(self.sortie(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dossier_patient = os.path.join(dossier_patients, patient)
chemin_ct       = os.path.join(dossier_patient, 'ct.nii')

# Préparation du volume CT (Normalisation identique à l'entraînement)
volume_ct = nib.load(chemin_ct).get_fdata().astype(np.float32)
volume_ct = np.clip(volume_ct, -200, 1800)
volume_ct = (volume_ct - (-200)) / (1800 - (-200))

# Chargement du masque "Vérité Terrain" pour calcul du Dice final
masque_verite = np.zeros(volume_ct.shape, dtype=np.uint8)
for nom_scapula in ['scapula_left', 'scapula_right']:
    chemin_masque = os.path.join(dossier_patient, 'segmentations', f'{nom_scapula}.nii')
    if os.path.exists(chemin_masque):
        masque_local = (nib.load(chemin_masque).get_fdata() > 0).astype(np.uint8)
        if masque_local.shape != volume_ct.shape:
            masque_local = np.transpose(masque_local, (2, 1, 0))
        if masque_local.shape == volume_ct.shape:
            masque_verite = np.clip(masque_verite + masque_local, 0, 1)

print(f"Voxels humérus vérité terrain : {masque_verite.sum():,}")

#Prédiction U-Net patch par patch
# On utilise un système d'accumulation pour gérer le chevauchement des patches.
modele = UNet3D()
poids  = torch.load(chemin_modele, map_location='cpu', weights_only=False) 
# Nettoyage des clés si le modèle a été entraîné avec DataParallel
if any(cle.startswith('module.') for cle in poids.keys()):
    poids = {cle.replace('module.', ''): valeur for cle, valeur in poids.items()}
modele.load_state_dict(poids)
modele = modele.to(device)
modele.eval()

carte_prediction = np.zeros(volume_ct.shape, dtype=np.float32)
compteur_patches = np.zeros(volume_ct.shape, dtype=np.float32)
taille_patch, pas = 64, 32

z_max, y_max, x_max = volume_ct.shape
with torch.no_grad():
    for z in range(0, z_max - taille_patch + 1, pas):
        for y in range(0, y_max - taille_patch + 1, pas):
            for x in range(0, x_max - taille_patch + 1, pas):
                patch_ct = volume_ct[z:z+taille_patch, y:y+taille_patch, x:x+taille_patch] 
                if patch_ct.shape != (taille_patch, taille_patch, taille_patch):
                    continue
                patch_tensor     = torch.tensor(patch_ct).unsqueeze(0).unsqueeze(0).float().to(device)  
                prediction_patch = modele(patch_tensor).squeeze().cpu().numpy()  
                carte_prediction[z:z+taille_patch, y:y+taille_patch, x:x+taille_patch] += prediction_patch 
                compteur_patches[z:z+taille_patch, y:y+taille_patch, x:x+taille_patch] += 1

# Moyennage des zones de chevauchement
compteur_sauvegarde                        = compteur_patches.copy()
compteur_patches[compteur_patches == 0]    = 1
carte_prediction                           = carte_prediction / compteur_patches
carte_prediction[compteur_sauvegarde == 0] = 0

# Post-traitement et Nettoyage
#Seuillage binaire
masque_binaire = (carte_prediction > 0.5).astype(np.uint8)

#Extraction de la plus grande composante connexe (élimination des faux positifs)
composants, nb_composants = label(masque_binaire)
tailles_composants        = ndimage.sum(masque_binaire, composants, range(1, nb_composants+1))
tailles_triees            = sorted(enumerate(tailles_composants), key=lambda x: x[1], reverse=True)

print("Top 5 composants :")
for rang, (idx, taille) in enumerate(tailles_triees[:5]):
    print(f"  #{rang+1} : {int(taille):,} voxels")

seuil_voxels  = 10000
labels_gardes = [idx+1 for idx, taille in enumerate(tailles_composants) if taille > seuil_voxels]
masque_final  = np.isin(composants, labels_gardes).astype(np.uint8)

#Remplissage des cavités internes (Binary Fill Holes)
masque_final  = binary_fill_holes(masque_final).astype(np.uint8)
print(f"Voxels finaux : {masque_final.sum():,}")

#Évaluation du score Dice
nb_voxels_communs = (masque_final * masque_verite).sum()
dice_score        = 2 * nb_voxels_communs / (masque_final.sum() + masque_verite.sum())
print(f"\n Dice score : {dice_score:.3f}")

#Génération du modèle 3D (STL)
#Utilisation de Marching Cubes pour transformer la grille de voxels en maillage 3D.
masque_lisse                = gaussian_filter(masque_final.astype(np.float32), sigma=1.0)
sommets, faces, normales, _ = marching_cubes(masque_lisse, level=0.4)
print(f"Sommets : {len(sommets):,} | Faces : {len(faces):,}")

#Export STL
def exporter_stl(sommets, faces, chemin_fichier):
    with open(chemin_fichier, 'wb') as fichier:
        fichier.write(b'\0' * 80) # Header
        fichier.write(struct.pack('<I', len(faces)))
        for face in faces:
            sommet_0 = sommets[face[0]]
            sommet_1 = sommets[face[1]]
            sommet_2 = sommets[face[2]]
            normale  = np.cross(sommet_1 - sommet_0, sommet_2 - sommet_0)
            norme    = np.linalg.norm(normale)
            if norme > 0:
                normale /= norme
            fichier.write(struct.pack('<3f', *normale))
            fichier.write(struct.pack('<3f', *sommet_0))
            fichier.write(struct.pack('<3f', *sommet_1))
            fichier.write(struct.pack('<3f', *sommet_2))
            fichier.write(struct.pack('<H', 0))
    print(f"STL exporté : {chemin_fichier}")

chemin_stl = dossier_sortie + f'scapula_{patient}.stl'
exporter_stl(sommets, faces, chemin_stl)
print(f"\n Terminé - Dice : {dice_score:.3f}")

Device : cuda
Patient : s0224
Volume CT : (333, 333, 621)
Voxels humérus vérité terrain : 90,676
Max prédiction : 1.000
% > 0.5 : 0.15%
% > 0.7 : 0.12%
Nettoyage...
Top 5 composants :
  #1 : 43,770 voxels
  #2 : 40,833 voxels
  #3 : 6,109 voxels
  #4 : 4,202 voxels
  #5 : 2,724 voxels
Voxels finaux : 84,603

🎯 Dice score : 0.909
Marching Cubes...
Sommets : 53,050 | Faces : 106,096
STL exporté : /kaggle/working/scapula_s0224.stl

✅ Terminé — Dice : 0.909


In [ ]:
#Étape 4 bis : Inférence complète et Post-traitement de l'humérus
# Utilisation du second modèle entraîné pour segmenter l'humérus.
# Le processus est identique à celui de la scapula mais utilise les poids spécifiques.
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
import os
import struct
from skimage.measure import marching_cubes
from scipy.ndimage import gaussian_filter, label, binary_fill_holes
import scipy.ndimage as ndimage

base_dir   = '/kaggle/input/datasets/raphalperon/data-scap/patients4/patients3/patients3'
model_path = '/kaggle/input/datasets/raphalperon/data-scap/meilleur_modele_humerus33.pth'
patient    = 's0224' 
output_dir = '/kaggle/working/'

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.encodeur = nn.ModuleList()
        self.pool     = nn.MaxPool3d(kernel_size=2, stride=2)
        for feature in features:
            self.encodeur.append(DoubleConv(in_channels, feature))
            in_channels = feature
        self.fond = DoubleConv(features[-1], features[-1] * 2)
        self.decodeur_up   = nn.ModuleList()
        self.decodeur_conv = nn.ModuleList()
        for feature in reversed(features):
            self.decodeur_up.append(
                nn.ConvTranspose3d(feature * 2, feature, kernel_size=2, stride=2)
            )
            self.decodeur_conv.append(DoubleConv(feature * 2, feature))
        self.sortie  = nn.Conv3d(features[0], out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        skip_connections = []
        for couche in self.encodeur:
            x = couche(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.fond(x)
        skip_connections = skip_connections[::-1]
        for i in range(len(self.decodeur_up)):
            x    = self.decodeur_up[i](x)
            skip = skip_connections[i]
            if x.shape != skip.shape:
                x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.decodeur_conv[i](x)
        return self.sigmoid(self.sortie(x))

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

dossier_path = os.path.join(base_dir, patient)
ct_path = os.path.join(dossier_path, 'ct.nii')

print(f"Patient : {patient}")
ct = nib.load(ct_path).get_fdata().astype(np.float32)
ct = np.clip(ct, -200, 1800)
ct = (ct - (-200)) / (1800 - (-200))
print(f"Volume CT : {ct.shape}")

# Charger masque vérité terrain pour calcul Dice
masque_global = np.zeros(ct.shape, dtype=np.uint8)
for nom in ['humerus_left', 'humerus_right']:
    path_masque = os.path.join(dossier_path, 'segmentations', f'{nom}.nii')
    if os.path.exists(path_masque):
        masque_local = (nib.load(path_masque).get_fdata() > 0).astype(np.uint8)
        if masque_local.shape != ct.shape:
            masque_local = np.transpose(masque_local, (2, 1, 0))
        if masque_local.shape == ct.shape:
            masque_global = np.clip(masque_global + masque_local, 0, 1)
        break

print(f"Voxels scapula GT : {masque_global.sum():,}")

# ── Étape 2 : Prédiction U-Net ────────────────────────────────
model = UNet3D()
poids = torch.load(path_masque, map_location='cpu',weights_only=False)
if any(cle.startswith('module.') for cle in poids.keys()):
    poids = {cle.replace('module.', ''): valeur for cle, valeur in poids.items()}
model.load_state_dict(poids)
model = model.to(device)
model.eval()

masque_prediction = np.zeros(ct.shape, dtype=np.float32)
compteur_patch    = np.zeros(ct.shape, dtype=np.float32)
taille, pas    = 64, 32

z_max, y_max, x_max = ct.shape
with torch.no_grad():
    for z in range(0, z_max - taille + 1, pas):
        for y in range(0, y_max - taille + 1, pas):
            for x in range(0, x_max - taille + 1, pas):
                patch_ct = ct[z:z+taille, y:y+taille, x:x+taille]
                if patch_ct.shape != (taille, taille, taille):
                    continue
                patch_tensor = torch.tensor(ct).unsqueeze(0).unsqueeze(0).float().to(device)
                prediction_patch = model(t).squeeze().cpu().numpy()
                masque_prediction[z:z+taille, y:y+taille, x:x+taille] += p
                compteur[z:z+taille,       y:y+taille, x:x+taille] += 1

compteur_orig                  = compteur.copy()
compteur[compteur == 0]        = 1
masque_complet                 = masque_complet / compteur
masque_complet[compteur_orig == 0] = 0

print(f"Max prédiction : {masque_complet.max():.3f}")
print(f"% > 0.5 : {(masque_complet > 0.5).mean()*100:.2f}%")
print(f"% > 0.7 : {(masque_complet > 0.7).mean()*100:.2f}%")

# ── Étape 3 : Nettoyage ───────────────────────────────────────
print("Nettoyage...")
masque_binaire = (masque_complet > 0.5).astype(np.uint8)

labels_cc, nb = label(masque_binaire)
tailles       = ndimage.sum(masque_binaire, labels_cc, range(1, nb+1))
tailles_triees = sorted(enumerate(tailles), key=lambda x: x[1], reverse=True)

print("Top 5 composants :")
for rang, (idx, t) in enumerate(tailles_triees[:5]):
    print(f"  #{rang+1} : {int(t):,} voxels")

seuil_taille = 10000
gros_labels  = [idx+1 for idx, t in enumerate(tailles) if t > seuil_taille]
masque_final = np.isin(labels_cc, gros_labels).astype(np.uint8)
masque_final    = binary_fill_holes(masque_final).astype(np.uint8)
print(f"Voxels finaux : {masque_final.sum():,}")

# ── Dice score ────────────────────────────────────────────────
intersection = (masque_final * masque_gt).sum()
dice = 2 * intersection / (masque_final.sum() + masque_gt.sum())
print(f"\n🎯 Dice score : {dice:.3f}")

# ── Étape 4 : Marching Cubes ──────────────────────────────────
print("Marching Cubes...")
masque_lisse             = gaussian_filter(masque_final.astype(np.float32), sigma=1.0)
vertices, faces, normals, _ = marching_cubes(masque_lisse, level=0.4)
print(f"Vertices : {len(vertices):,} | Faces : {len(faces):,}")

# ── Étape 5 : Export STL ──────────────────────────────────────
def exporter_stl(vertices, faces, filename):
    with open(filename, 'wb') as f:
        f.write(b'\0' * 80)
        f.write(struct.pack('<I', len(faces)))
        for face in faces:
            v0, v1, v2 = vertices[face[0]], vertices[face[1]], vertices[face[2]]
            normal = np.cross(v1 - v0, v2 - v0)
            norm   = np.linalg.norm(normal)
            if norm > 0:
                normal /= norm
            f.write(struct.pack('<3f', *normal))
            f.write(struct.pack('<3f', *v0))
            f.write(struct.pack('<3f', *v1))
            f.write(struct.pack('<3f', *v2))
            f.write(struct.pack('<H', 0))
    print(f"STL exporté : {filename}")

stl_path = output_dir + f'humerus_{patient}.stl'
exporter_stl(vertices, faces, stl_path)
print(f"\n✅ Terminé — Dice : {dice:.3f}")

In [1]:
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
import os
import struct
from skimage.measure import marching_cubes
from scipy.ndimage import gaussian_filter, label, binary_fill_holes
import scipy.ndimage as ndimage

#Chemins
dossier_patient_scap = '/kaggle/input/datasets/raphalperon/data-scap/patients4/patients3/patients3'
dossier_patient_hum  = '/kaggle/input/datasets/raphalperon/data-scap/patient/patient'
patient          = 's0224'
dossier_sortie   = '/kaggle/working/'

#Dictionnaire de configuration pour la boucle
config_os = {
    'scapula': {
        'modele': '/kaggle/input/datasets/raphalperon/data-scap/meilleur_modele_scapula2.pth',
        'labels_gt': ['scapula_left', 'scapula_right']
    },
    'humerus': {
        'modele': '/kaggle/input/datasets/raphalperon/data-scap/meilleur_modele_humerus33.pth',
        'labels_gt': ['humerus_left', 'humerus_right']
    }
}

#Modèle
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.encodeur = nn.ModuleList()
        self.pool     = nn.MaxPool3d(kernel_size=2, stride=2)
        for feature in features:
            self.encodeur.append(DoubleConv(in_channels, feature))
            in_channels = feature
        self.fond = DoubleConv(features[-1], features[-1] * 2)
        self.decodeur_up   = nn.ModuleList()
        self.decodeur_conv = nn.ModuleList()
        for feature in reversed(features):
            self.decodeur_up.append(nn.ConvTranspose3d(feature * 2, feature, kernel_size=2, stride=2))
            self.decodeur_conv.append(DoubleConv(feature * 2, feature))
        self.sortie  = nn.Conv3d(features[0], out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        skip_connections = []
        for couche in self.encodeur:
            x = couche(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.fond(x)
        skip_connections = skip_connections[::-1]
        for i in range(len(self.decodeur_up)):
            x    = self.decodeur_up[i](x)
            skip = skip_connections[i]
            if x.shape != skip.shape:
                x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.decodeur_conv[i](x)
        return self.sigmoid(self.sortie(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device} | Patient : {patient}")

dossier_patient = os.path.join(dossier_patients, patient)
chemin_ct       = os.path.join(dossier_patient, 'ct.nii')

# Préparation du volume CT unique
volume_ct = nib.load(chemin_ct).get_fdata().astype(np.float32)
volume_ct = np.clip(volume_ct, -200, 1800)
volume_ct = (volume_ct - (-200)) / (1800 - (-200))
z_max, y_max, x_max = volume_ct.shape

#Fonction d'Export STL
def exporter_stl(sommets, faces, chemin_fichier):
    with open(chemin_fichier, 'wb') as fichier:
        fichier.write(b'\0' * 80)
        fichier.write(struct.pack('<I', len(faces)))
        for face in faces:
            s0, s1, s2 = sommets[face[0]], sommets[face[1]], sommets[face[2]]
            normale = np.cross(s1 - s0, s2 - s0)
            norme = np.linalg.norm(normale)
            if norme > 0: normale /= norme
            fichier.write(struct.pack('<3f', *normale))
            fichier.write(struct.pack('<3f', *s0))
            fichier.write(struct.pack('<3f', *s1))
            fichier.write(struct.pack('<3f', *s2))
            fichier.write(struct.pack('<H', 0))

#Boucle d'inférence (scapula puis humérus)
for nom_os, config in config_os.items():
    print(f"\n--- Traitement de l'os : {nom_os.upper()} ---")

    #Chargement de la Vérité Terrain (GT)
    masque_verite = np.zeros(volume_ct.shape, dtype=np.uint8)
    for label_name in config['labels_gt']:
        path_gt = os.path.join(dossier_patient, 'segmentations', f'{label_name}.nii')
        if os.path.exists(path_gt):
            m_local = (nib.load(path_gt).get_fdata() > 0).astype(np.uint8)
            if m_local.shape != volume_ct.shape:
                m_local = np.transpose(m_local, (2, 1, 0))
            masque_verite = np.clip(masque_verite + m_local, 0, 1)
    
    #Prédiction U-Net patch par patch
    modele = UNet3D().to(device)
    poids = torch.load(config['modele'], map_location=device, weights_only=False)
    # On utilise un système d'accumulation pour gérer le chevauchement des patches.
    if any(cle.startswith('module.') for cle in poids.keys()):
        poids = {cle.replace('module.', ''): valeur for cle, valeur in poids.items()}
    modele.load_state_dict(poids)
    modele.eval()

    carte_prediction = np.zeros(volume_ct.shape, dtype=np.float32)
    compteur_patches = np.zeros(volume_ct.shape, dtype=np.float32)
    taille, pas = 64, 32

    with torch.no_grad():
        for z in range(0, z_max - taille + 1, pas):
            for y in range(0, y_max - taille + 1, pas):
                for x in range(0, x_max - taille + 1, pas):
                    patch_ct = volume_ct[z:z+taille, y:y+taille, x:x+taille]
                    patch_tensor = torch.tensor(patch_ct).unsqueeze(0).unsqueeze(0).float().to(device)
                    pred_patch = modele(patch_tensor).squeeze().cpu().numpy()
                    
                    carte_prediction[z:z+taille, y:y+taille, x:x+taille] += pred_patch
                    compteur_patches[z:z+taille, y:y+taille, x:x+taille] += 1

    # Moyennage
    c_sauv = compteur_patches.copy()
    compteur_patches[compteur_patches == 0] = 1
    carte_prediction /= compteur_patches
    carte_prediction[c_sauv == 0] = 0

    #Post-traitement et Nettoyage
    masque_binaire = (carte_prediction > 0.5).astype(np.uint8)
    labels_cc, nb = label(masque_binaire)
    tailles = ndimage.sum(masque_binaire, labels_cc, range(1, nb+1))
    
    seuil_voxels = 10000
    labels_gardes = [idx+1 for idx, t in enumerate(tailles) if t > seuil_voxels]
    masque_final = np.isin(labels_cc, labels_gardes).astype(np.uint8)
    #Remplissage des cavités internes (Binary Fill Holes)
    masque_final = binary_fill_holes(masque_final).astype(np.uint8)

    #Évaluation Dice
    intersection = (masque_final * masque_verite).sum()
    dice = 2 * intersection / (masque_final.sum() + masque_verite.sum())
    print(f"Dice Score {nom_os} : {dice:.3f}")

    #Génération du modèle 3D (STL)
    #Utilisation de Marching Cubes pour transformer la grille de voxels en maillage 3D.
    masque_lisse = gaussian_filter(masque_final.astype(np.float32), sigma=1.0)
    sommets, faces, _, _ = marching_cubes(masque_lisse, level=0.4)
    
    chemin_stl = os.path.join(dossier_sortie, f'{nom_os}_{patient}.stl')
    exporter_stl(sommets, faces, chemin_stl)
    print(f"Fichier STL généré : {chemin_stl}")

print("\n Inférence terminée pour toutes les structures.")

Device : cuda | Patient : s0224

--- Traitement de l'os : SCAPULA ---
Dice Score scapula : 0.909
Fichier STL généré : /kaggle/working/scapula_s0224.stl

--- Traitement de l'os : HUMERUS ---
Dice Score humerus : 0.000
Fichier STL généré : /kaggle/working/humerus_s0224.stl

 Inférence terminée pour toutes les structures.


In [2]:
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
import os
import struct
from skimage.measure import marching_cubes
from scipy.ndimage import gaussian_filter, label, binary_fill_holes
import scipy.ndimage as ndimage

# Chemins de base
patient = 's0224'
dossier_sortie = '/kaggle/working/'

# Configuration : On définit ici le dossier racine propre à chaque os
config_os = {
    'scapula': {
        'modele': '/kaggle/input/datasets/raphalperon/data-scap/meilleur_modele_scapula2.pth',
        'labels_gt': ['scapula_left', 'scapula_right'],
        'chemin_racine': '/kaggle/input/datasets/raphalperon/data-scap/patients4/patients3/patients3'
    },
    'humerus': {
        'modele': '/kaggle/input/datasets/raphalperon/data-scap/meilleur_modele_humerus33.pth',
        'labels_gt': ['humerus_left', 'humerus_right'],
        'chemin_racine': '/kaggle/input/datasets/raphalperon/data-scap/patient/patient' # Ton nouveau path
    }
}

# --- Modèle UNet3D (identique) ---
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.block(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.encodeur = nn.ModuleList()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        for feature in features:
            self.encodeur.append(DoubleConv(in_channels, feature))
            in_channels = feature
        self.fond = DoubleConv(features[-1], features[-1] * 2)
        self.decodeur_up = nn.ModuleList()
        self.decodeur_conv = nn.ModuleList()
        for feature in reversed(features):
            self.decodeur_up.append(nn.ConvTranspose3d(feature * 2, feature, kernel_size=2, stride=2))
            self.decodeur_conv.append(DoubleConv(feature * 2, feature))
        self.sortie = nn.Conv3d(features[0], out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        skip_connections = []
        for couche in self.encodeur:
            x = couche(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.fond(x)
        skip_connections = skip_connections[::-1]
        for i in range(len(self.decodeur_up)):
            x = self.decodeur_up[i](x)
            skip = skip_connections[i]
            if x.shape != skip.shape: x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.decodeur_conv[i](x)
        return self.sigmoid(self.sortie(x))

# --- Préparation CT ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# On utilise le dossier scapula pour charger le CT (qui est le même pour les deux)
chemin_ct = os.path.join(config_os['scapula']['chemin_racine'], patient, 'ct.nii')
volume_ct = nib.load(chemin_ct).get_fdata().astype(np.float32)
volume_ct = np.clip(volume_ct, -200, 1800)
volume_ct = (volume_ct - (-200)) / (1800 - (-200))
z_max, y_max, x_max = volume_ct.shape

def exporter_stl(sommets, faces, chemin_fichier):
    with open(chemin_fichier, 'wb') as fichier:
        fichier.write(b'\0' * 80)
        fichier.write(struct.pack('<I', len(faces)))
        for face in faces:
            s0, s1, s2 = sommets[face[0]], sommets[face[1]], sommets[face[2]]
            normale = np.cross(s1 - s0, s2 - s0)
            norme = np.linalg.norm(normale)
            if norme > 0: normale /= norme
            fichier.write(struct.pack('<3f', *normale))
            fichier.write(struct.pack('<3f', *s0))
            fichier.write(struct.pack('<3f', *s1))
            fichier.write(struct.pack('<3f', *s2))
            fichier.write(struct.pack('<H', 0))

# --- Boucle d'inférence ---
for nom_os, config in config_os.items():
    print(f"\n--- Traitement : {nom_os.upper()} ---")

    # MODIF ICI : Utilise le chemin spécifique défini dans la config
    dossier_seg = os.path.join(config['chemin_racine'], patient, 'segmentations')
    
    masque_verite = np.zeros(volume_ct.shape, dtype=np.uint8)
    for label_name in config['labels_gt']:
        path_gt = os.path.join(dossier_seg, f'{label_name}.nii')
        if os.path.exists(path_gt):
            m_local = (nib.load(path_gt).get_fdata() > 0).astype(np.uint8)
            if m_local.shape != volume_ct.shape: m_local = np.transpose(m_local, (2, 1, 0))
            masque_verite = np.clip(masque_verite + m_local, 0, 1)

    # Inférence
    modele = UNet3D().to(device)
    poids = torch.load(config['modele'], map_location=device, weights_only=False)
    if any(cle.startswith('module.') for cle in poids.keys()):
        poids = {cle.replace('module.', ''): v for cle, v in poids.items()}
    modele.load_state_dict(poids)
    modele.eval()

    carte_prediction = np.zeros(volume_ct.shape, dtype=np.float32)
    compteur_patches = np.zeros(volume_ct.shape, dtype=np.float32)
    taille, pas = 64, 32

    with torch.no_grad():
        for z in range(0, z_max - taille + 1, pas):
            for y in range(0, y_max - taille + 1, pas):
                for x in range(0, x_max - taille + 1, pas):
                    patch_ct = volume_ct[z:z+taille, y:y+taille, x:x+taille]
                    patch_tensor = torch.tensor(patch_ct).unsqueeze(0).unsqueeze(0).float().to(device)
                    pred_patch = modele(patch_tensor).squeeze().cpu().numpy()
                    carte_prediction[z:z+taille, y:y+taille, x:x+taille] += pred_patch
                    compteur_patches[z:z+taille, y:y+taille, x:x+taille] += 1

    carte_prediction /= np.maximum(compteur_patches, 1)
    masque_binaire = (carte_prediction > 0.5).astype(np.uint8)
    
    # Nettoyage
    labels_cc, nb = label(masque_binaire)
    tailles = ndimage.sum(masque_binaire, labels_cc, range(1, nb+1))
    labels_gardes = [idx+1 for idx, t in enumerate(tailles) if t > 10000]
    masque_final = np.isin(labels_cc, labels_gardes).astype(np.uint8)
    masque_final = binary_fill_holes(masque_final).astype(np.uint8)

    # Dice
    intersection = (masque_final * masque_verite).sum()
    dice = 2 * intersection / (masque_final.sum() + masque_verite.sum())
    print(f"Dice Score {nom_os} : {dice:.3f}")

    # STL
    masque_lisse = gaussian_filter(masque_final.astype(np.float32), sigma=1.0)
    sommets, faces, _, _ = marching_cubes(masque_lisse, level=0.4)
    chemin_stl = os.path.join(dossier_sortie, f'{nom_os}_{patient}.stl')
    exporter_stl(sommets, faces, chemin_stl)

print("\nTerminé.")


--- Traitement : SCAPULA ---
Dice Score scapula : 0.909

--- Traitement : HUMERUS ---
Dice Score humerus : 0.964

Terminé.


In [3]:
#Étape 5 : Assemblage Anatomique (Scapula + Humérus)
# Cette étape fusionne la scapula et l'humérus prédite par le modèle
# pour obtenir une vue complète de l'articulation de l'épaule.
patient = 's0224'
def lire_stl(chemin):
    """Lire un STL binaire et retourner vertices et faces"""
    with open(chemin, 'rb') as f:
        f.read(80)  # Saut du header (80 octets)
        nb_faces = struct.unpack('<I', f.read(4))[0]
        vertices = []
        faces    = []
        for i in range(nb_faces):
            f.read(12)  # Saut du vecteur normale
            v0 = struct.unpack('<3f', f.read(12))
            v1 = struct.unpack('<3f', f.read(12))
            v2 = struct.unpack('<3f', f.read(12))
            f.read(2)   # Saut de l'attribut (2 octets)
            idx = len(vertices)
            vertices.extend([v0, v1, v2])
            faces.append([idx, idx+1, idx+2])
    return np.array(vertices), np.array(faces)

# Chargement des deux structures
vertices_scapula, faces_scapula = lire_stl(f'/kaggle/working/scapula_{patient}.stl')
vertices_humerus, faces_humerus = lire_stl(f'/kaggle/input/datasets/raphalperon/data-scap/humerus_{patient}.stl')

# Fusion des maillages : décalage des index des faces de l'humérus
offset                 = len(vertices_scapula)
faces_humerus_decalees = faces_humerus + offset

# Concaténation finale
vertices_final = np.concatenate([vertices_scapula, vertices_humerus], axis=0)
faces_final    = np.concatenate([faces_scapula,    faces_humerus_decalees], axis=0)

print(f"Scapula  : {len(vertices_scapula):,} vertices | {len(faces_scapula):,} faces")
print(f"Humérus  : {len(vertices_humerus):,} vertices | {len(faces_humerus):,} faces")
print(f"Total    : {len(vertices_final):,} vertices | {len(faces_final):,} faces")

# Export du modèle complet
exporter_stl(vertices_final, faces_final, f'/kaggle/working/epaule_{patient}.stl')
print(f"\n Assemblage terminé - epaule_{patient}.stl")

Scapula  : 318,288 vertices | 106,096 faces
Humérus  : 218,952 vertices | 72,984 faces
Total    : 537,240 vertices | 179,080 faces

 Assemblage terminé — epaule_s0224.stl
